# MCTS Demo

Step-by-step walkthrough of Monte Carlo Tree Search on Bridgit with the 1-2-2 turn structure.

## 1. Setup — create net, wrapper, MCTS

In [ ]:
import numpy as np
from bridgit import Bridgit, Player
from bridgit.ai import BridgitNet, NetWrapper, MCTS, Config, GameWrapper

N = 3  # small board for fast demo
config = Config(board_size=N, num_mcts_sims=50)
net = BridgitNet(board_size=N, num_channels=32, num_res_blocks=2)
wrapper = NetWrapper(net)
mcts = MCTS(wrapper, config)
gw = GameWrapper(N)

print(f"Board: {config.grid_size}x{config.grid_size}")
print(f"Action space: {config.action_size} (flat), {len(Bridgit(N).get_available_moves())} legal crossings")
print(f"Device: {wrapper.device}")

## 2. Raw neural net output (before any search)

The untrained net gives random-ish policy and value. Let's see what it produces on an empty board.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

game = Bridgit(N)
g = config.grid_size

# Get raw net prediction
policy, value = mcts._predict(game)
policy_2d = policy.reshape(g, g)

# Mask to only legal crossings
mask = gw.get_valid_moves_mask(game).reshape(g, g)
masked_policy = policy_2d * mask
masked_policy /= masked_policy.sum()

fig = make_subplots(rows=1, cols=2, subplot_titles=["Raw Policy (all cells)", "Masked Policy (legal moves only)"])

fig.add_trace(go.Heatmap(z=np.flipud(policy_2d), colorscale="Blues", showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=np.flipud(masked_policy), colorscale="Reds", showscale=False), row=1, col=2)

fig.update_layout(width=700, height=350, title=f"Net value estimate: {value:.3f}")
fig.show()

## 3. Single MCTS search — visit counts and action probs

Run 50 simulations on the empty board. Compare raw policy (net alone) vs MCTS policy (net + search).

In [ ]:
game = Bridgit(N)

# MCTS search
visit_counts = mcts.search(game)
mcts_probs = mcts.get_action_probs(game, temperature=1.0)

visits_2d = visit_counts.reshape(g, g)
mcts_2d = mcts_probs.reshape(g, g)

fig = make_subplots(rows=1, cols=2, subplot_titles=["Visit Counts", "MCTS Policy (temp=1)"])

fig.add_trace(go.Heatmap(z=np.flipud(visits_2d), colorscale="Greens", showscale=True), row=1, col=1)
fig.add_trace(go.Heatmap(z=np.flipud(mcts_2d), colorscale="Reds", showscale=True), row=1, col=2)

fig.update_layout(width=700, height=350, title="MCTS concentrates visits on promising moves")
fig.show()

# Print top moves
print(f"Player: {game.current_player.name}, moves_left: {game.moves_left_in_turn}")
print(f"\nTop 5 moves by visit count:")
legal_actions = np.nonzero(gw.get_valid_moves_mask(game))[0]
sorted_actions = sorted(legal_actions, key=lambda a: visit_counts[a], reverse=True)
for a in sorted_actions[:5]:
    r, c = gw.action_to_move(a)
    print(f"  ({r},{c})  visits={int(visit_counts[a]):3d}  prob={mcts_probs[a]:.3f}")

## 4. Verify 1-2-2 turn structure in the tree

Inspect the root and first two levels. Root is P1 with 1 move → children are P2 with 2 moves → grandchildren are P2 with 1 move (same player!).

In [ ]:
from bridgit.ai.mcts import MCTSNode

# Build a fresh tree so we can inspect it
game = Bridgit(N)
root = MCTSNode(game.copy())
mcts._expand(root)

# Run a few sims to populate the tree
for _ in range(config.num_mcts_sims):
    node = root
    while node.is_expanded and node.children:
        node = node.best_child(config.c_puct)
    if node.game.game_over:
        mcts._backpropagate(node, 1.0)
        continue
    value = mcts._expand(node)
    mcts._backpropagate(node, value)

# Inspect tree structure
print(f"ROOT: player={root.game.current_player.name}, moves_left={root.game.moves_left_in_turn}")
print(f"  children: {len(root.children)}\n")

# Pick the most visited child
best_action = max(root.children, key=lambda a: root.children[a].visit_count)
child = root.children[best_action]
r, c = gw.action_to_move(best_action)
print(f"CHILD (move {r},{c}): player={child.game.current_player.name}, moves_left={child.game.moves_left_in_turn}")
print(f"  visits={child.visit_count}, Q={child.q_value:.3f}")
print(f"  children: {len(child.children)}")

if child.children:
    # Pick most visited grandchild
    best_gc = max(child.children, key=lambda a: child.children[a].visit_count)
    gc = child.children[best_gc]
    r2, c2 = gw.action_to_move(best_gc)
    print(f"\nGRANDCHILD (move {r2},{c2}): player={gc.game.current_player.name}, moves_left={gc.game.moves_left_in_turn}")
    print(f"  visits={gc.visit_count}, Q={gc.q_value:.3f}")
    print(f"  children: {len(gc.children)}")

    # Verify: child and grandchild should be the SAME player (2-move turn)
    same = child.game.current_player == gc.game.current_player
    print(f"\n>>> Child and grandchild same player? {same} (expected: True for 2-move turn)")

## 5. Full game — MCTS vs MCTS

Play a complete game, visualizing each board state. Each player uses the same untrained net.

In [ ]:
game = Bridgit(N)
history = []

while not game.game_over:
    player = game.current_player
    left = game.moves_left_in_turn
    probs = mcts.get_action_probs(game, temperature=0.5)
    action = np.random.choice(len(probs), p=probs)
    row, col = gw.action_to_move(action)
    game.make_move(row, col)
    history.append((row, col, player.name, left))
    print(f"Move {len(history):2d}: ({row},{col}) by {player.name:10s}  (had {left} left in turn)")

print(f"\nWinner: {game.winner.name} in {len(history)} moves")

In [ ]:
# Visualize the final board
game.state.visualize()

## 6. Replay — visualize each move

Replay the game move by move, showing the board after each turn (pair of moves).

In [ ]:
# Replay: show board after each player's turn ends (when moves_left was 1)
from bridgit.game import GameState

replay = Bridgit(N)
for i, (row, col, player_name, moves_left) in enumerate(history):
    replay.make_move(row, col)
    # Show board when a turn ends (moves_left was 1) or game is over
    if moves_left == 1 or replay.game_over:
        print(f"After move {i+1} ({row},{col}) by {player_name}:")
        print(replay.state)
        print()

## 7. Backpropagation sanity check

Verify that value signs are correct: same-player consecutive nodes should accumulate the same sign, different-player transitions flip.

In [ ]:
# Walk the most-visited path from root to a leaf and check Q values
game = Bridgit(N)
root = MCTSNode(game.copy())
mcts._expand(root)

for _ in range(config.num_mcts_sims):
    node = root
    while node.is_expanded and node.children:
        node = node.best_child(config.c_puct)
    if node.game.game_over:
        mcts._backpropagate(node, 1.0)
        continue
    value = mcts._expand(node)
    mcts._backpropagate(node, value)

# Trace the most-visited path
print("Most-visited path from root:\n")
print(f"{'Depth':>5} {'Move':>8} {'Player':>12} {'MovesLeft':>10} {'Visits':>7} {'Q':>7} {'Same as parent?':>16}")
print("-" * 75)

node = root
depth = 0
print(f"{depth:>5} {'root':>8} {node.game.current_player.name:>12} {node.game.moves_left_in_turn:>10} {node.visit_count:>7} {node.q_value:>7.3f} {'':>16}")

while node.children:
    best_a = max(node.children, key=lambda a: node.children[a].visit_count)
    child = node.children[best_a]
    depth += 1
    r, c = gw.action_to_move(best_a)
    same = "YES" if child.game.current_player == node.game.current_player else "no (flip)"
    print(f"{depth:>5} {f'({r},{c})':>8} {child.game.current_player.name:>12} {child.game.moves_left_in_turn:>10} {child.visit_count:>7} {child.q_value:>7.3f} {same:>16}")
    node = child
    if depth > 8:
        print("  ...")
        break